# Detector verification harness

Every "verified on synthetic data" claim in `EXPERIMENTS.md` (rows 4-9) was produced
by ad hoc, throwaway code -- there is no committed, re-runnable version of it anywhere
in this repo (confirmed by grepping `baseline_with_viz.ipynb` for "synthetic",
"permutation", "ablation", "GroupKFold": zero hits in code, only in markdown prose and
in `EXPERIMENTS.md` itself). This notebook is that missing harness: the four canonical
synthetic scenarios used informally throughout the experiment log, made concrete and
reusable, plus the specific checks run before adding the river-backed features
(`EXPERIMENTS.md`, Stage A).

This notebook is deliberately **separate** from `baseline_with_viz.ipynb` (the actual
competition submission) and does not import from it -- partly to keep the submission
notebook's own top-to-bottom reproduction time from growing with every new candidate
verified here (see `EXPERIMENTS.md` §3's reproduction steps), and partly because it is
not established that `crunch`'s notebook-to-submission packaging would carry along a
second local file if one tried to import it. The small amount of overlap (a detector's
hyperparameters) is kept in sync by hand and called out explicitly wherever it matters
-- see the note in the ADWIN/PageHinkley section below.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from river import drift

## Synthetic scenarios

Four scenarios, matching what rows 4-9 of `EXPERIMENTS.md` describe informally:
mean shift, variance shift, autocorrelation shift, and a no-break null (for
false-positive-rate checks). Each returns a historical segment, an online segment,
and causal binary labels (`1` from `tau` onward) exactly like the real
`(x_hist, x_online, tau)` triples `train()`/`infer()` see.

In [ ]:
def make_scenario(kind: str, n_hist: int = 300, n_online: int = 300, tau: int = 150, seed: int = 0):
    """
    kind: "no_break", "mean_shift", "variance_shift", or "autocorr_shift".
    Returns (x_hist, x_online, labels) -- labels is all-zero for "no_break".
    """
    r = np.random.default_rng(seed)
    x_hist = r.normal(0, 1, n_hist)

    if kind == "no_break":
        x_online = r.normal(0, 1, n_online)
    elif kind == "mean_shift":
        x_online = np.concatenate([r.normal(0, 1, tau), r.normal(2.5, 1, n_online - tau)])
    elif kind == "variance_shift":
        x_online = np.concatenate([r.normal(0, 1, tau), r.normal(0, 3, n_online - tau)])
    elif kind == "autocorr_shift":
        pre = r.normal(0, 1, tau)
        post = np.zeros(n_online - tau)
        prev = 0.0
        for i in range(len(post)):
            prev = 0.8 * prev + r.normal(0, 0.6)
            post[i] = prev
        x_online = np.concatenate([pre, post])
    else:
        raise ValueError(f"unknown scenario kind: {kind!r}")

    labels = np.zeros(n_online, dtype=int)
    if kind != "no_break":
        labels[tau:] = 1
    return x_hist, x_online, labels


def run_synthetic_battery(score_fn, n_trials: int = 10, **scenario_kwargs) -> dict:
    """
    Runs `score_fn(x_hist, x_online) -> np.ndarray of per-point scores` across
    the three break scenarios (reporting mean AUC over `n_trials` seeds each)
    plus the no-break null (reporting the mean *final* score, as a
    false-positive/no-break-floor indicator -- there's no positive class to
    compute an AUC against for a pure null).
    """
    results = {}
    for kind in ["mean_shift", "variance_shift", "autocorr_shift"]:
        aucs = []
        for trial in range(n_trials):
            x_hist, x_online, labels = make_scenario(kind, seed=trial, **scenario_kwargs)
            scores = score_fn(x_hist, x_online)
            if 0 < labels.sum() < len(labels):
                aucs.append(roc_auc_score(labels, scores))
        results[kind] = float(np.mean(aucs)) if aucs else float("nan")

    floor_scores = []
    for trial in range(n_trials):
        x_hist, x_online, _ = make_scenario("no_break", seed=trial + 1000, **scenario_kwargs)
        floor_scores.append(float(score_fn(x_hist, x_online)[-1]))
    results["no_break_floor"] = float(np.mean(floor_scores))
    return results

## Stage A: river's ADWIN + PageHinkley

Fed the same way `baseline_with_viz.ipynb`'s `_RiverDetectorBank` feeds them (ADWIN
gets `z_raw`, river's PageHinkley gets raw `x`) -- if either detector's hyperparameters
change in the submission notebook, mirror the change here too, since this cell
hardcodes river's own defaults rather than importing `baseline_with_viz.ipynb`'s
constants (see the top-of-notebook note on why these two notebooks don't share code
directly).

In [ ]:
def adwin_variance_scores(x_hist, x_online):
    """ADWIN's own adaptive-window variance, fed z_raw -- not a flag, a continuous stat."""
    mu_h, sd_h = x_hist.mean(), max(x_hist.std(ddof=1), 1e-8)
    a = drift.ADWIN()
    out = []
    for x in x_online:
        a.update((x - mu_h) / sd_h)
        out.append(a.variance)
    return np.array(out)


def adwin_n_detections_scores(x_hist, x_online):
    mu_h, sd_h = x_hist.mean(), max(x_hist.std(ddof=1), 1e-8)
    a = drift.ADWIN()
    out = []
    for x in x_online:
        a.update((x - mu_h) / sd_h)
        out.append(float(a.n_detections))
    return np.array(out)


def river_ph_n_flags_scores(x_hist, x_online):
    """river's PageHinkley fed raw x (no historical reference), matching the
    existing hand-rolled Tier-2 page_hinkley's own convention."""
    ph = drift.PageHinkley()
    out = []
    n_flags = 0
    for x in x_online:
        ph.update(float(x))
        if ph.drift_detected:
            n_flags += 1
        out.append(float(n_flags))
    return np.array(out)


print("ADWIN .variance (continuous):", run_synthetic_battery(adwin_variance_scores))
print("ADWIN .n_detections (cumulative flag count):", run_synthetic_battery(adwin_n_detections_scores))
print("river PageHinkley n_flags (cumulative flag count):", run_synthetic_battery(river_ph_n_flags_scores))

**Result** (see `EXPERIMENTS.md` for the version tied to an actual commit): ADWIN's
adaptive-window `.variance` is a strong discriminator on its own -- variance-shift AUC
~1.0, and notably ~0.95 on the autocorrelation-shift scenario, better than every other
method's synthetic autocorrelation number logged in `EXPERIMENTS.md` §5 (CUSUM 0.815,
`feature` 0.964, Page-Hinkley 0.59). ADWIN's discrete `.n_detections` count, by
contrast, sits at ~0.50 across every scenario -- it essentially never fires within a
300-point online window regardless of scenario, so on its own it carries almost no
signal; it's kept as a feature in the GBM anyway (free to compute, and the tree can
ignore it if it's not pulling weight, exactly as `cusum`/`page_hinkley` were kept as
raw features in Tier 3 despite ranking low in permutation importance). river's
PageHinkley (cumulative flag count, no continuous statistic available) does
respectably on mean-shift (~0.95, expected -- it's fundamentally a mean-shift test)
but is not clearly better than the existing hand-rolled Tier-2 `page_hinkley` (row 7:
0.996 mean-shift, 0.75 variance-shift) -- a fair, single-variable comparison of the
EWMA-forgetting variant against the non-forgetting one, per the design review that
flagged the input-transform confound risk.

**Scale-invariance check** (ADWIN is fed `z_raw`; confirms the affine rescaling itself
introduces no scale-dependent false-positive behavior at two very different native
scales):

In [ ]:
for std in [1, 50]:
    r = np.random.default_rng(42)
    x_hist = r.normal(0, std, 300)
    x_online = r.normal(0, std, 300)
    mu_h, sd_h = x_hist.mean(), x_hist.std(ddof=1)
    a = drift.ADWIN()
    for x in x_online:
        a.update((x - mu_h) / sd_h)
    print(f"std={std}: n_detections after 300 no-break points = {a.n_detections}")

## KSWIN: benchmarked and dropped from Stage A

`river.drift.KSWIN` was the other Stage-A candidate in the original plan. Benchmarked
here **before** wiring it into the submission notebook, per the design-review finding
that it calls `scipy`'s `ks_2samp` on every single `.update()` call once its window
fills (unlike ADWIN, whose expensive check is gated behind `clock=32`) -- so its cost
does not scale down with a smaller window the way one might expect.

In [ ]:
import time

vals = np.random.default_rng(0).normal(0, 1, 200_000)

for label, ws, ss in [("default 100/30", 100, 30), ("small 50/15", 50, 15), ("tiny 30/10", 30, 10)]:
    k = drift.KSWIN(window_size=ws, stat_size=ss, seed=0)
    t0 = time.perf_counter()
    for v in vals:
        k.update(float(v))
    t1 = time.perf_counter()
    per_point_us = (t1 - t0) / len(vals) * 1e6
    projected_min = per_point_us * 1e-6 * 10_000 * 505 / 60  # ~10,000 training series, ~505 mean online length
    print(f"KSWIN({label}): {per_point_us:.1f} us/point -> projected train() cost: {projected_min:.1f} min")

a = drift.ADWIN()
t0 = time.perf_counter()
for v in vals:
    a.update(float(v))
t1 = time.perf_counter()
per_point_us = (t1 - t0) / len(vals) * 1e6
print(f"ADWIN: {per_point_us:.3f} us/point -> projected: {per_point_us * 1e-6 * 10_000 * 505 / 60:.3f} min (for comparison)")

**Result**: ~650-690 microseconds/point regardless of window size -- the cost is
dominated by the per-call `ks_2samp` + `random.sample()` overhead, not by how much data
those calls process. Projects to **~55-58 minutes added to `train()` for this one
detector alone**, more than double the entire wavelet/spectral feature family's cost
(`EXPERIMENTS.md` row 9: ~19-20 min), against a track record where every feature
family added so far has bought +0.001 to +0.005 TS-AUC. ADWIN, by contrast, projects
to a couple of seconds across the whole training set.

**Decision**: KSWIN is dropped from Stage A. Not abandoned -- revisit if ADWIN/
PageHinkley's real result justifies the added budget, or if a cheaper variant (e.g.
clocked/subsampled, checking only every N points the way ADWIN's own `clock`
parameter does) is worth building. This mirrors how the original plan treated ADWIN,
BOCPD, and the two-state regime-switching model in `EXPERIMENTS.md` §1: measured,
deprioritized with a stated reason, not silently dropped.

## Binarization proxy for the DDM/EDDM/HDDM family (Stage C -- not yet implemented)

Placeholder for the next stage: `river.drift.binary.{DDM,EDDM,HDDMA,HDDMW}` need a
binary "error" stream, which this dataset does not naturally have. Before wiring any
of them in, whichever proxy is chosen (`|z_raw| > threshold` vs. thresholding `z_std`
directly, per the design review's point that variance shifts are this dataset's
dominant signal) needs its false-positive rate checked here against the `no_break`
scenario above -- consecutive "error" bits derived from a rolling statistic are
autocorrelated across overlapping windows, which risks biasing these detectors toward
overconfidence (declaring drift on what looks like independent accumulating evidence
but is really redundant overlapping windows).